****Read Bronze Tables****

Read raw posts and comments data from the Bronze layer tables.

These tables contain unprocessed log data used as input for transformation.

In [0]:
posts_bronze= spark.table("log_catalog.bronze.posts")

In [0]:
comments_bronze  = spark.table("log_catalog.bronze.comments")


parsing logs using pyspark


***IMPORT FUNCTION FOR SELECTING COLUMNS***

Import required function to reference and manipulate DataFrame columns.
Select only required columns from posts data and rename id to postId.
Helps reduce data size and avoid column ambiguity during joins.

In [0]:
from pyspark.sql.functions import col

In [0]:
posts_df = posts_bronze.select(col("id").alias("postId"),col("title"))

In [0]:
comments_df = comments_bronze.select(col("id").alias("commentId"),col("postId"),col("email").alias("user"), col("body").alias("comment"))

***Join DataFrames***

Join comments with posts using postId to combine related data.

In [0]:
silver_df = comments_df.join(posts_df,on="postId",how="inner")

***Remove Null Values***

Remove records with null values in important columns.

In [0]:
silver_df = silver_df.dropna(subset=["postId", "commentId", "comment", "user"])

***REMOVED DUPLICATES***
Remove duplicate records based on unique commentId.

In [0]:
silver_df= silver_df.dropDuplicates(["commentId"])

**invalid data filtered**

Filter out invalid records where postId is not valid.

In [0]:
silver_df = silver_df.filter(col("postId") > 0)

***Import Additional Functions***

Import functions for conditional logic and timestamp generation.
Used for creating derived columns in the dataset.

In [0]:
from pyspark.sql.functions import when, current_timestamp

***Add Log Level Column***

Create a new column log_level based on comment content

Categorizes logs into ERROR or INFO for better analysis.

In [0]:
silver_df = silver_df.withColumn( "log_level", when(col("comment").contains("error"), "ERROR").otherwise("INFO"))


In [0]:
silver_df = silver_df.withColumn("timestamp",current_timestamp())

In [0]:
display(silver_df)

postId,commentId,user,comment,title,log_level,timestamp
1,1,Eliseo@gardner.biz,laudantium enim quasi est quidem magnam voluptate ipsam eos tempora quo necessitatibus dolor quam autem quasi reiciendis et nam sapiente accusantium,sunt aut facere repellat provident occaecati excepturi optio reprehenderit,INFO,2026-04-05T17:40:12.159Z
1,2,Jayne_Kuhic@sydney.com,est natus enim nihil est dolore omnis voluptatem numquam et omnis occaecati quod ullam at voluptatem error expedita pariatur nihil sint nostrum voluptatem reiciendis et,sunt aut facere repellat provident occaecati excepturi optio reprehenderit,ERROR,2026-04-05T17:40:12.159Z
1,3,Nikita@garfield.biz,quia molestiae reprehenderit quasi aspernatur aut expedita occaecati aliquam eveniet laudantium omnis quibusdam delectus saepe quia accusamus maiores nam est cum et ducimus et vero voluptates excepturi deleniti ratione,sunt aut facere repellat provident occaecati excepturi optio reprehenderit,INFO,2026-04-05T17:40:12.159Z
1,4,Lew@alysha.tv,non et atque occaecati deserunt quas accusantium unde odit nobis qui voluptatem quia voluptas consequuntur itaque dolor et qui rerum deleniti ut occaecati,sunt aut facere repellat provident occaecati excepturi optio reprehenderit,INFO,2026-04-05T17:40:12.159Z
1,5,Hayden@althea.biz,harum non quasi et ratione tempore iure ex voluptates in ratione harum architecto fugit inventore cupiditate voluptates magni quo et,sunt aut facere repellat provident occaecati excepturi optio reprehenderit,INFO,2026-04-05T17:40:12.159Z
2,6,Presley.Mueller@myrl.com,doloribus at sed quis culpa deserunt consectetur qui praesentium accusamus fugiat dicta voluptatem rerum ut voluptate autem voluptatem repellendus aspernatur dolorem in,qui est esse,INFO,2026-04-05T17:40:12.159Z
2,7,Dallas@ole.me,maiores sed dolores similique labore et inventore et quasi temporibus esse sunt id et eos voluptatem aliquam aliquid ratione corporis molestiae mollitia quia et magnam dolor,qui est esse,INFO,2026-04-05T17:40:12.159Z
2,8,Mallory_Kunze@marie.org,ut voluptatem corrupti velit ad voluptatem maiores et nisi velit vero accusamus maiores voluptates quia aliquid ullam eaque,qui est esse,INFO,2026-04-05T17:40:12.159Z
2,9,Meghan_Littel@rene.us,sapiente assumenda molestiae atque adipisci laborum distinctio aperiam et ab ut omnis et occaecati aspernatur odit sit rem expedita quas enim ipsam minus,qui est esse,INFO,2026-04-05T17:40:12.159Z
2,10,Carmen_Keeling@caroline.name,voluptate iusto quis nobis reprehenderit ipsum amet nulla quia quas dolores velit et non aut quia necessitatibus nostrum quaerat nulla et accusamus nisi facilis,qui est esse,INFO,2026-04-05T17:40:12.159Z


structure 

In [0]:
silver_df = silver_df.select("postId","commentId","comment","user","log_level","timestamp")

**Write to Data Lake**

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://log-data@stlogprocessing123.dfs.core.windows.net/silver/comments")

**SAVE AS TABLE**

In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("log_catalog.silver.comments_silver")

In [0]:
silver_df.printSchema()

root
 |-- postId: long (nullable = true)
 |-- commentId: long (nullable = true)
 |-- comment: string (nullable = true)
 |-- user: string (nullable = true)
 |-- log_level: string (nullable = false)
 |-- timestamp: timestamp (nullable = false)



In [0]:
silver_df.write.format("delta") \
    .mode("overwrite") \
         .option("overwriteSchema", "true") \
    .saveAsTable("log_catalog.silver.comments")

In [0]:
# %sql SHOW TABLES IN log_catalog.silver;

In [0]:
%sql
SELECT * FROM log_catalog.silver.comments LIMIT 10;

postId,commentId,comment,user,log_level,timestamp
6,28,voluptatem repellendus quo alias at laudantium mollitia quidem esse temporibus consequuntur vitae rerum illum id corporis sit id,Ronny@rosina.org,INFO,2026-04-05T17:41:34.759Z
6,29,tempora voluptatem est magnam distinctio autem est dolorem et ipsa molestiae odit rerum itaque corporis nihil nam eaque rerum error,Jennings_Pouros@erica.biz,ERROR,2026-04-05T17:41:34.759Z
6,30,consequuntur quia voluptate assumenda et autem voluptatem reiciendis ipsum animi est provident earum aperiam sapiente ad vitae iste accusantium aperiam eius qui dolore voluptatem et,Lurline@marvin.biz,INFO,2026-04-05T17:41:34.759Z
7,33,fugit harum quae vero libero unde tempore soluta eaque culpa sequi quibusdam nulla id et et necessitatibus,Jaeden.Towne@arlene.tv,INFO,2026-04-05T17:41:34.759Z
19,93,ut quis et id repellat labore nobis itaque quae saepe est ullam aut dolor id ut quis sunt iure voluptates expedita voluptas doloribus modi saepe autem,Roma_Doyle@alia.com,INFO,2026-04-05T17:41:34.759Z
31,151,quos eos sint voluptatibus similique iusto perferendis omnis voluptas earum nulla cumque dolorem consequatur officiis quis consequatur aspernatur nihil ullam et enim enim unde nihil labore non ducimus,Cary@taurean.biz,INFO,2026-04-05T17:41:34.759Z
35,171,sed non beatae placeat qui libero nam eaque qui quia ut ad doloremque sequi unde quidem adipisci debitis autem velit architecto aperiam eos nihil enim dolores veritatis omnis ex,Gerda.Reynolds@ceasar.co.uk,INFO,2026-04-05T17:41:34.759Z
40,200,facere maxime alias aspernatur ab quibusdam necessitatibus ratione similique error enim sed minus et et provident minima voluptatibus,Amir@kaitlyn.org,ERROR,2026-04-05T17:41:34.759Z
42,208,enim aut doloremque mollitia provident molestiae omnis esse excepturi officiis tempora sequi molestiae veniam voluptatem recusandae omnis temporibus et deleniti laborum sed ipsa molestiae eum aut,Cesar_Volkman@letitia.biz,INFO,2026-04-05T17:41:34.759Z
45,221,rerum possimus asperiores non dolores maiores tenetur ab suscipit laudantium possimus ab iure distinctio assumenda iste adipisci optio est sed eligendi temporibus perferendis tempore soluta,Madonna@will.com,INFO,2026-04-05T17:41:34.759Z


In [0]:
post_comments_silver = spark.table("log_catalog.silver.comments_silver")

In [0]:
post_comments_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("log_catalog.silver.post_comments_silver")

In [0]:
df = spark.table("log_catalog.silver.comments_silver")

display(df)

postId,commentId,comment,user,log_level,timestamp
6,28,voluptatem repellendus quo alias at laudantium mollitia quidem esse temporibus consequuntur vitae rerum illum id corporis sit id,Ronny@rosina.org,INFO,2026-04-05T17:40:23.405Z
6,29,tempora voluptatem est magnam distinctio autem est dolorem et ipsa molestiae odit rerum itaque corporis nihil nam eaque rerum error,Jennings_Pouros@erica.biz,ERROR,2026-04-05T17:40:23.405Z
6,30,consequuntur quia voluptate assumenda et autem voluptatem reiciendis ipsum animi est provident earum aperiam sapiente ad vitae iste accusantium aperiam eius qui dolore voluptatem et,Lurline@marvin.biz,INFO,2026-04-05T17:40:23.405Z
7,33,fugit harum quae vero libero unde tempore soluta eaque culpa sequi quibusdam nulla id et et necessitatibus,Jaeden.Towne@arlene.tv,INFO,2026-04-05T17:40:23.405Z
19,93,ut quis et id repellat labore nobis itaque quae saepe est ullam aut dolor id ut quis sunt iure voluptates expedita voluptas doloribus modi saepe autem,Roma_Doyle@alia.com,INFO,2026-04-05T17:40:23.405Z
31,151,quos eos sint voluptatibus similique iusto perferendis omnis voluptas earum nulla cumque dolorem consequatur officiis quis consequatur aspernatur nihil ullam et enim enim unde nihil labore non ducimus,Cary@taurean.biz,INFO,2026-04-05T17:40:23.405Z
35,171,sed non beatae placeat qui libero nam eaque qui quia ut ad doloremque sequi unde quidem adipisci debitis autem velit architecto aperiam eos nihil enim dolores veritatis omnis ex,Gerda.Reynolds@ceasar.co.uk,INFO,2026-04-05T17:40:23.405Z
40,200,facere maxime alias aspernatur ab quibusdam necessitatibus ratione similique error enim sed minus et et provident minima voluptatibus,Amir@kaitlyn.org,ERROR,2026-04-05T17:40:23.405Z
42,208,enim aut doloremque mollitia provident molestiae omnis esse excepturi officiis tempora sequi molestiae veniam voluptatem recusandae omnis temporibus et deleniti laborum sed ipsa molestiae eum aut,Cesar_Volkman@letitia.biz,INFO,2026-04-05T17:40:23.405Z
45,221,rerum possimus asperiores non dolores maiores tenetur ab suscipit laudantium possimus ab iure distinctio assumenda iste adipisci optio est sed eligendi temporibus perferendis tempore soluta,Madonna@will.com,INFO,2026-04-05T17:40:23.405Z


In [0]:
df.show(10, truncate=False)

+------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+---------+--------------------------+
|postId|commentId|comment                                                                                                                                                                                                    |user                       |log_level|timestamp                 |
+------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------+---------+--------------------------+
|6     |28       |voluptatem repellendus quo alias at laudantium\nmollitia quidem esse\ntemporibus consequuntur vitae rerum illum\nid co

In [0]:
df.printSchema()

root
 |-- postId: long (nullable = true)
 |-- commentId: long (nullable = true)
 |-- comment: string (nullable = true)
 |-- user: string (nullable = true)
 |-- log_level: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [0]:
df.columns

In [0]:
df.dtypes

In [0]:
df.describe().show()